In [9]:
import os

# import arxiv
import ast
import concurrent
from csv import writer
from IPython.display import display, Markdown, Latex
import json
import openai
import os
import pandas as pd
# from PyPDF2 import PdfReader
import requests
from scipy import spatial
from tenacity import retry, wait_random_exponential, stop_after_attempt
# import tiktoken
from tqdm import tqdm
from termcolor import colored

GPT_MODEL = "gpt-3.5-turbo-0613"
EMBEDDING_MODEL = "text-embedding-ada-002"


In [ ]:
@retry(wait=wait_random_exponential(min=1, max=40), stop=stop_after_attempt(3))
def embedding_request(text):
    response = openai.Embedding.create(input=text, model=EMBEDDING_MODEL)
    return response

def strings_ranked_by_relatedness(
    query: str,
    df: pd.DataFrame,
    relatedness_fn=lambda x, y: 1 - spatial.distance.cosine(x, y),
    top_n: int = 100,
) -> list[str]:
    """Returns a list of strings and relatednesses, sorted from most related to least."""
    query_embedding_response = embedding_request(query)
    query_embedding = query_embedding_response["data"][0]["embedding"]
    strings_and_relatednesses = [
        (row["filepath"], relatedness_fn(query_embedding, row["embedding"]))
        for i, row in df.iterrows()
    ]
    strings_and_relatednesses.sort(key=lambda x: x[1], reverse=True)
    strings, relatednesses = zip(*strings_and_relatednesses)
    return strings[:top_n]


input_text = 'Basketball Game'
response = embedding_request(input_text)
response

In [68]:
from openai import OpenAI
client = OpenAI()

path = "/Users/brianlerner/Library/CloudStorage/OneDrive-Personal/Code/Cerebra/src/llm/basketball_practice_name_variants.csv"
df = pd.read_csv(path)
# names = df.name
# # df.set_index('name', inplace=True)
# embeddings = [get_embedding(e) for e in names]
# df = pd.DataFrame(embeddings, index=names)


In [69]:
def get_embedding(text, model="text-embedding-ada-002"):
   # text = text.replace("\n", " ") # not important for v2 models apparently
   return client.embeddings.create(input = [text], model=model).data[0].embedding

df["embeddings"] = df.name.apply(lambda x: get_embedding(x, model='text-embedding-ada-002'))
# embeddings = df.index.apply(lambda x: get_embedding(x, model='text-embedding-ada-002'))
# df.to_csv('output/embedded_1k_reviews.csv', index=False)

In [70]:
def get_closest_variable_names(base_name, variable_names):
    from scipy.spatial.distance import cosine 

    df = pd.DataFrame(variable_names, columns=["name"])

    df["distances"] = df["embeddings"].apply(lambda x: cosine(x, df.iloc[0]["embeddings"])) 
    df.sort_values(by="distances", inplace=True)

    return df["name"].tolist()[0]


,name,embeddings,distances
0,basketball practice,"[-0.02714117057621479, -0.018223358318209648, ...",0.000000
10,basketball workout,"[-0.038629673421382904, -0.013945875689387321,...",0.048529
1,basketball training,"[-0.03401206061244011, -0.023811085149645805, ...",0.054415
7,ball practice,"[-0.026611868292093277, -0.007630857173353434,...",0.054607
13,basketball session,"[-0.02749904990196228, -0.015430746600031853, ...",0.055804
12,basketball rehearsal,"[-0.02532731369137764, -0.013601201586425304, ...",0.064246
14,sports practice,"[-0.012956863269209862, -0.002043988322839141,...",0.064570
5,basketball drills,"[-0.04561406001448631, -0.0234640222042799, 0....",0.065597
19,basketball scrimmage,"[-0.038355953991413116, -0.016618287190794945,...",0.076404
6,shooting practice,"[-0.02835761569440365, -0.002552387537434697, ...",0.079821


In [ ]:

df= df[df.Score!=3]
df['sentiment'] = df.Score.replace({1:'negative', 2:'negative', 4:'positive', 5:'positive'})

labels = ['negative', 'positive']
label_embeddings = [get_embedding(label, model=model) for label in labels]

def label_score(review_embedding, label_embeddings):
   return cosine_similarity(review_embedding, label_embeddings[1]) - cosine_similarity(review_embedding, label_embeddings[0])

prediction = 'positive' if label_score('Sample Review', label_embeddings) > 0 else 'negative'